# Modelado: preprocesamiento + selección de variables × 4 modelos

Un estudio de Optuna **por modelo** (ridge, lasso, mlp, xgboost). Cada trial elige a la vez los
hiperparámetros del modelo y la configuración del pipeline:

`transformador` → `scaler` → `selección de variables` → `modelo`

- Se evalúa con `TimeSeriesSplit` **sobre train**; después de cada fold se reporta el MAE medio acumulado y el
  `MedianPruner` corta los trials que van peor que la mediana en ese fold (no se desperdician folds en configs malas).
- La mejor configuración se reentrena en todo train y se mide en **validation**.
- Al final se compara contra los baselines (persistencia y media móvil 7 días), medidos en los mismos folds y en val.
- Test **no se carga** en este notebook.

Después del estudio general hay una **búsqueda exhaustiva** solo para lasso y MLP (los más prometedores), el
**registro de modelos** (`models/model_registry.json`, versiones `vX.Y.Z`) y la **comparación de residuos**.

`dataset_final.csv` ya viene con inválidos, outliers (IQR clip) e imputación KNN aplicados (`src/main.py`);
la transformación, el escalado y la selección se ajustan dentro de cada fold, acá.

## Configuración

In [ ]:
from __future__ import annotations

import json
import platform
import sys
import time
import warnings
from datetime import datetime
from pathlib import Path
from typing import Callable

import joblib
import matplotlib.pyplot as plt
import numpy as np
import optuna
import pandas as pd
import sklearn
from optuna.visualization import matplotlib as optuna_plt
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.exceptions import ConvergenceWarning
from sklearn.feature_selection import RFE, SelectFromModel
from sklearn.linear_model import Lasso, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (MinMaxScaler, PowerTransformer, QuantileTransformer,
                                   RobustScaler, StandardScaler)
from xgboost import XGBRegressor

# raíz del proyecto = la primera carpeta (desde acá hacia arriba) que tiene src/preprocess.py
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "preprocess.py").exists())
sys.path.insert(0, str(ROOT / "src"))
from preprocess import (COLS_CICLICAS, COLS_SIN_LIMITES, RUTA_FINAL, TARGET, cargar_registro,
                        guardar_registro, huella)
from feature_selection import SelectorFiltro, memoria   # memoria: caché en results/.cache

# ------------------------------------------------------------------ estudio general
MODELOS = ["ridge", "lasso", "mlp", "xgboost"]
N_TRIALS = 60        # trials por modelo
N_SPLITS = 5         # folds de TimeSeriesSplit sobre train
TIMEOUT = 20 * 60    # segundos máximos por modelo (None = sin límite)
SEMILLA = 42

# ------------------------------------------------------------------ búsqueda exhaustiva (lasso y MLP)
MODELOS_EXH = ["lasso", "mlp"]
N_TRIALS_EXH = 150
TIMEOUT_EXH = 30 * 60

# ------------------------------------------------------------------ rutas
RUTA_LOGS = ROOT / "results" / "experiment_logs.csv"
RUTA_OPTUNA_DB = ROOT / "results" / "optuna.db"   # estudios persistidos: optuna-dashboard la puede abrir
RUTA_MODELOS = ROOT / "models"                    # .joblib de cada versión registrada
RUTA_REGISTRO_MODELOS = RUTA_MODELOS / "model_registry.json"
RUTA_OPTUNA_DB.parent.mkdir(parents=True, exist_ok=True)

optuna.logging.set_verbosity(optuna.logging.WARNING)
warnings.filterwarnings("ignore", category=optuna.exceptions.ExperimentalWarning)

## Datos

In [ ]:
def cargar_datos(ruta: str | Path = RUTA_FINAL, incluir_test: bool = False) -> pd.DataFrame:
    """Lee dataset_final.csv indexado por fecha. Por defecto descarta las filas de test:
    ni siquiera quedan en memoria. Pasar incluir_test=True solo para la evaluación final."""
    df = pd.read_csv(ruta, parse_dates=["fecha"]).sort_values("fecha").set_index("fecha")
    if not incluir_test:
        df = df[df["split"] != "test"]
    return df


def particion(df: pd.DataFrame, split: str) -> tuple[pd.DataFrame, pd.Series]:
    """(X, y) de un split. Features = todo menos el target y la columna `split`."""
    d = df[df["split"] == split]
    return d.drop(columns=[TARGET, "split"]), d[TARGET]


def _metricas(y, pred) -> dict:
    y, pred = np.asarray(y, dtype=float), np.asarray(pred, dtype=float)
    ok = ~np.isnan(pred)  # los primeros días de un rolling quedan en NaN
    y, pred = y[ok], pred[ok]
    return {
        "mae": float(mean_absolute_error(y, pred)),
        "rmse": float(np.sqrt(mean_squared_error(y, pred))),
        "r2": float(r2_score(y, pred)),
    }

## Baselines

In [ ]:
def predicciones_baselines(df: pd.DataFrame) -> pd.DataFrame:
    """Predicción de cada baseline para la fila D (que apunta al consumo de D+1).

    El target de la fila D es frio(D+1), así que frio(D) = target de la fila D-1:
      persistencia   : frio(D)                      = y.shift(1)
      media móvil 7d : promedio de frio(D-6 .. D)   = y.shift(1).rolling(7).mean()
    Solo usa información disponible en D: no hay fuga hacia el futuro."""
    if not (df.index.to_series().diff().dropna() == pd.Timedelta(days=1)).all():
        raise ValueError("Hay días faltantes en dataset_final.csv: shift(1) ya no equivale a 'ayer'.")
    y = df[TARGET]
    return pd.DataFrame({
        "baseline_persistencia": y.shift(1),
        "baseline_media_movil_7d": y.shift(1).rolling(7).mean(),
    })

## Pipeline

| Paso | Opciones que prueba Optuna |
|---|---|
| Transformador | `yeo-johnson` / `quantile` (a normal) / `ninguno` |
| Scaler | `standard` / `robust` / `minmax` / `ninguno` |
| Selección | `filtro` (top-k por \|corr\| o información mutua, sin redundantes) / `filtro+rfe` (wrapper sobre las k del filtro) / `lasso` (embebido: columnas con coeficiente ≠ 0) / `ninguna` |
| Variables externas | **siempre entran**: no pasan por la selección |

**Variables externas** = clima (temperatura y humedad), mes (`mes_sin`, `mes_cos`) y producción (`hl cerveza envasada`),
con sus lags y medias móviles (21 columnas). El filtro es univariado: comparadas una a una contra el target, estas
variables siempre pierden contra el frío de hoy y sus medias móviles (muy autocorrelacionadas), así que nunca pasaban
la selección. Pero son las que explican los *cambios* de consumo: probando a mano, el MLP con las 5 variables de frío
más clima/producción/mes bajó de 2303 a 2241 de MAE val. Por eso van directo al modelo y la selección se aplica solo al
resto (`k` y `n_rfe` cuentan solo esas otras columnas). Las interacciones (`temp max (°c) x frio (kw)`, ...) no cuentan
como externas: siguen compitiendo en la selección.

`mes_sin` y `mes_cos` no se transforman ni escalan (ya están en [-1, 1]). Todo se ajusta solo con la parte de
entrenamiento de cada fold. El RFE usa el mismo modelo que se evalúa, salvo el MLP (no expone coeficientes) que usa Ridge.

`SelectorFiltro` vive en `src/feature_selection.py` (no acá) para que los modelos guardados con joblib se puedan
cargar desde cualquier script. Para no recalcular: el pipeline tiene `memory` (un mismo transformador + scaler en el
mismo fold se ajusta una sola vez entre trials) y los scores del filtro también se cachean.

In [ ]:
_TRANSFORMADORES = {
    "yeo-johnson": lambda: PowerTransformer(method="yeo-johnson", standardize=False),
    # n_quantiles < filas del fold más chico (~420 con 5 folds)
    "quantile": lambda: QuantileTransformer(n_quantiles=200, output_distribution="normal", random_state=SEMILLA),
}
_SCALERS = {"standard": StandardScaler, "robust": RobustScaler, "minmax": MinMaxScaler}

# clima y mes (COLS_SIN_LIMITES de preprocess) + producción, con sus lags y medias móviles
PREFIJOS_EXTERNOS = (*COLS_SIN_LIMITES, "hl cerveza envasada")


def columnas_externas(columnas) -> list[str]:
    """Variables externas que siempre entran al modelo. Las interacciones (" x ") y los ratios (" / ")
    no cuentan: mezclan con variables internas (p. ej. `temp max (°c) x frio (kw)`)."""
    return [c for c in columnas if str(c).startswith(PREFIJOS_EXTERNOS) and " x " not in c and " / " not in c]


def construir_pipeline(estimador, columnas, cfg: dict, rfe_estimador=None) -> Pipeline:
    """transformador -> scaler (menos mes_sin/mes_cos) -> selección (solo internas; las externas pasan
    directo) -> modelo."""
    num = [c for c in columnas if c not in COLS_CICLICAS]
    pasos_num = []
    if cfg["transformador"] != "ninguno":
        pasos_num.append(("trans", _TRANSFORMADORES[cfg["transformador"]]()))
    if cfg["scaler"] != "ninguno":
        pasos_num.append(("scaler", _SCALERS[cfg["scaler"]]()))
    prep = ColumnTransformer(
        [("num", Pipeline(pasos_num) if pasos_num else "passthrough", num)],
        remainder="passthrough", verbose_feature_names_out=False,
    ).set_output(transform="pandas")

    sel = cfg["seleccion"]
    filtro = (SelectorFiltro(cfg["filtro_metodo"], cfg["k"], cfg["umbral_redundancia"])
              if sel in ("filtro", "filtro+rfe") else "passthrough")
    if sel == "filtro+rfe":
        wrapper = RFE(rfe_estimador, n_features_to_select=cfg["n_rfe"], step=0.1)
    elif sel == "lasso":
        wrapper = SelectFromModel(Lasso(alpha=cfg["alpha_sel"], max_iter=20_000, random_state=SEMILLA))
    else:
        wrapper = "passthrough"

    externas = columnas_externas(columnas)
    internas = [c for c in columnas if c not in externas]
    if sel == "ninguna":
        seleccion = "passthrough"
    else:
        # la selección solo ve las internas; las externas se suman sin tocar (ColumnTransformer le pasa y al filtro/RFE)
        seleccion = ColumnTransformer(
            [("sel", Pipeline([("filtro", filtro), ("wrapper", wrapper)]), internas),
             ("externas", "passthrough", externas)],
            verbose_feature_names_out=False,
        )
    pipe = Pipeline([("prep", prep), ("seleccion", seleccion), ("modelo", estimador)], memory=memoria)
    return pipe.set_output(transform="pandas")


def columnas_elegidas(pipe: Pipeline) -> list[str]:
    """Columnas que llegan al modelo (después de todos los pasos de selección)."""
    return list(pipe[:-1].get_feature_names_out())

## Espacios de búsqueda

In [ ]:
def _mlp(**params) -> TransformedTargetRegressor:
    """Red neuronal (MLP) con el target estandarizado.

    El target está en kW (~10000) y los pesos de la red arrancan cerca de 0: sin estandarizarlo
    la red entrena muy mal. El escalado de y se ajusta solo con los datos de entrenamiento que
    recibe .fit() y viaja dentro del objeto, así que .predict() devuelve kW directamente."""
    return TransformedTargetRegressor(regressor=MLPRegressor(**params), transformer=StandardScaler())


# Optuna no acepta tuplas como opciones categóricas: se elige un nombre y se traduce.
_CAPAS_MLP = {"32": (32,), "64": (64,), "64-32": (64, 32), "128-64": (128, 64)}


def _espacio_ridge(t: optuna.Trial) -> dict:
    return {"alpha": t.suggest_float("alpha", 1e-2, 1e4, log=True)}


def _espacio_lasso(t: optuna.Trial) -> dict:
    return {"alpha": t.suggest_float("alpha", 1e-1, 1e3, log=True)}


def _espacio_xgb(t: optuna.Trial) -> dict:
    return {
        "n_estimators": t.suggest_int("n_estimators", 200, 800, step=100),
        "learning_rate": t.suggest_float("learning_rate", 0.01, 0.1, log=True),
        "max_depth": t.suggest_int("max_depth", 2, 6),
        "subsample": t.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": t.suggest_float("colsample_bytree", 0.5, 1.0),
        "reg_lambda": t.suggest_float("reg_lambda", 1.0, 100.0, log=True),
        "min_child_weight": t.suggest_int("min_child_weight", 1, 5),
    }


def _espacio_mlp(t: optuna.Trial) -> dict:
    return {
        "hidden_layer_sizes": _CAPAS_MLP[t.suggest_categorical("hidden_layer_sizes", list(_CAPAS_MLP))],
        "activation": t.suggest_categorical("activation", ["relu", "tanh"]),
        "alpha": t.suggest_float("alpha", 1e-4, 10.0, log=True),   # regularización L2
        "learning_rate_init": t.suggest_float("learning_rate_init", 1e-3, 1e-2, log=True),
        "batch_size": t.suggest_categorical("batch_size", [32, 64, 128]),
    }


ESPACIOS: dict[str, dict] = {
    "ridge": {"clase": Ridge, "fijos": {"random_state": SEMILLA}, "espacio": _espacio_ridge},
    # max_iter acotado: con alpha chico Lasso tarda muchísimo en converger y el MAE decide igual
    "lasso": {"clase": Lasso, "fijos": {"random_state": SEMILLA, "max_iter": 20_000}, "espacio": _espacio_lasso},
    "xgboost": {"clase": XGBRegressor,
                "fijos": {"random_state": SEMILLA, "n_jobs": -1, "tree_method": "hist"},
                "espacio": _espacio_xgb},
    "mlp": {
        "clase": _mlp,
        # Sin early_stopping: scikit-learn lo haría con un 10 % ALEATORIO de train, que en una
        # serie temporal mezcla pasado y futuro. Corta por n_iter_no_change sobre la pérdida de train.
        "fijos": {"random_state": SEMILLA, "solver": "adam", "max_iter": 1000,
                  "n_iter_no_change": 20, "early_stopping": False},
        "espacio": _espacio_mlp,
    },
}


def espacio_prep(t: optuna.Trial, k=(10, 150, 10), frac_rfe=(0.2, 0.9), n_rfe_min=5,
                 alpha_sel=(1e-1, 1e3)) -> dict:
    """Configuración del pipeline. Los parámetros de cada método de selección solo se sugieren
    si ese método sale elegido. Los rangos son argumentos para que la búsqueda exhaustiva los amplíe."""
    cfg = {
        "transformador": t.suggest_categorical("transformador", ["yeo-johnson", "quantile", "ninguno"]),
        "scaler": t.suggest_categorical("scaler", ["standard", "robust", "minmax", "ninguno"]),
        "seleccion": t.suggest_categorical("seleccion", ["filtro", "filtro+rfe", "lasso", "ninguna"]),
    }
    if cfg["seleccion"] in ("filtro", "filtro+rfe"):
        cfg["filtro_metodo"] = t.suggest_categorical("filtro_metodo", ["pearson", "spearman", "mutual_info"])
        cfg["k"] = t.suggest_int("k", *k[:2], step=k[2])
        cfg["umbral_redundancia"] = t.suggest_float("umbral_redundancia", 0.80, 1.0)  # 1.0 = sin filtro de redundancia
    if cfg["seleccion"] == "filtro+rfe":
        # RFE siempre después del filtro: correrlo sobre las 390 columnas es carísimo
        cfg["n_rfe"] = max(n_rfe_min, int(cfg["k"] * t.suggest_float("frac_rfe", *frac_rfe)))
    if cfg["seleccion"] == "lasso":
        cfg["alpha_sel"] = t.suggest_float("alpha_sel", *alpha_sel, log=True)
    return cfg


def sugerir(t: optuna.Trial, nombre: str, espacios: dict = ESPACIOS) -> tuple[dict, dict]:
    """(hiperparámetros del modelo, configuración del pipeline). Con un FixedTrial reconstruye la mejor.
    Cada espacio puede traer su propio "prep"; si no, usa espacio_prep con los rangos por defecto."""
    esp = espacios[nombre]
    return {**esp["fijos"], **esp["espacio"](t)}, esp.get("prep", espacio_prep)(t)


def ajustar(nombre: str, params: dict, cfg: dict, X: pd.DataFrame, y: pd.Series,
            espacios: dict = ESPACIOS) -> Pipeline:
    clase = espacios[nombre]["clase"]
    rfe_est = Ridge() if nombre == "mlp" else clase(**params)  # el MLP no tiene coef_ ni feature_importances_
    pipe = construir_pipeline(clase(**params), X.columns, cfg, rfe_est)
    with warnings.catch_warnings():
        # Lasso/MLP avisan "no convergió" y Yeo-Johnson desborda en columnas enormes; el MAE decide igual.
        warnings.simplefilter("ignore", ConvergenceWarning)
        warnings.simplefilter("ignore", RuntimeWarning)
        warnings.simplefilter("ignore", UserWarning)
        return pipe.fit(X, y)

## Registro de experimentos

In [ ]:
COLUMNAS_LOG = [
    "corrida", "timestamp", "modelo",
    "estudio",           # nombre del estudio de Optuna (modelo_etiqueta_corrida)
    "etapa",             # baseline | optuna_cv | optuna_val
    "estado",            # completo | podado (trial de Optuna cortado por el pruner o que falló)
    "hiperparametros",   # del modelo
    "preprocesamiento",  # transformador, scaler y selección
    "mae_cv", "std_cv", "n_folds",             # optuna_cv, optuna_val (mejor trial) y baseline
    "mae_val", "rmse_val", "r2_val", "mae_train",
    "seg_entrenamiento", "n_features", "n_train", "n_val", "semilla", "dataset_sha256",
]


def registrar_experimento(fila: dict, ruta: str | Path = RUTA_LOGS) -> None:
    """Agrega una fila a results/experiment_logs.csv (lo crea si no existe).
    Si el log viene de una versión anterior (otras columnas), lo completa
    con las columnas nuevas sin perder las filas viejas."""
    ruta = Path(ruta)
    ruta.parent.mkdir(parents=True, exist_ok=True)
    nueva = pd.DataFrame([{c: fila.get(c, np.nan) for c in COLUMNAS_LOG}])
    if not ruta.exists():
        nueva.to_csv(ruta, index=False)
        return
    if list(pd.read_csv(ruta, nrows=0).columns) == COLUMNAS_LOG:
        nueva.to_csv(ruta, mode="a", header=False, index=False)
    else:
        previo = pd.read_csv(ruta)
        extra = [c for c in previo.columns if c not in COLUMNAS_LOG]
        pd.concat([previo, nueva], ignore_index=True).reindex(columns=COLUMNAS_LOG + extra).to_csv(ruta, index=False)

## Optimización con Optuna (con pruning)

In [ ]:
def _objetivo(nombre: str, X_tr: pd.DataFrame, y_tr: pd.Series, cv: TimeSeriesSplit,
              anotar: Callable, espacios: dict = ESPACIOS) -> Callable[[optuna.Trial], float]:
    """MAE medio sobre los folds de TimeSeriesSplit (solo train). Después de cada fold reporta
    el MAE medio acumulado; si el pruner decide que el trial no promete, se corta ahí.
    Una config inválida (p. ej. la selección deja 0 columnas) también se poda."""
    def objetivo(trial: optuna.Trial) -> float:
        params, cfg = sugerir(trial, nombre, espacios)
        maes: list[float] = []
        n_feat = None
        t0 = time.perf_counter()
        for paso, (i_fit, i_eval) in enumerate(cv.split(X_tr)):
            try:
                pipe = ajustar(nombre, params, cfg, X_tr.iloc[i_fit], y_tr.iloc[i_fit], espacios)
                pred = pipe.predict(X_tr.iloc[i_eval])
            except ValueError as e:
                trial.set_user_attr("error", str(e)[:200])
                anotar(params, cfg, maes or [np.nan], time.perf_counter() - t0, "podado", n_feat)
                raise optuna.TrialPruned() from e
            n_feat = len(columnas_elegidas(pipe))
            trial.set_user_attr("n_features", n_feat)
            maes.append(mean_absolute_error(y_tr.iloc[i_eval], pred))
            trial.report(float(np.mean(maes)), step=paso)
            if trial.should_prune():
                anotar(params, cfg, maes, time.perf_counter() - t0, "podado", n_feat)
                raise optuna.TrialPruned()
        trial.set_user_attr("std_cv", float(np.std(maes)))
        anotar(params, cfg, maes, time.perf_counter() - t0, "completo", n_feat)
        return float(np.mean(maes))
    return objetivo


def comparar_modelos(
    modelos: list[str] = MODELOS,
    n_trials: int = N_TRIALS,
    n_splits: int = N_SPLITS,
    timeout: float | None = TIMEOUT,
    espacios: dict = ESPACIOS,
    etiqueta: str = "general",
    encolar: dict[str, dict] | None = None,
    n_startup: int = 8,
    ruta_datos: str | Path = RUTA_FINAL,
    ruta_logs: str | Path = RUTA_LOGS,
    verbose: bool = True,
) -> tuple[pd.DataFrame, dict[str, optuna.Study], dict[str, dict]]:
    """Baselines + un estudio de Optuna por modelo (TPE + MedianPruner, semilla fija).
    `encolar[modelo]`: parámetros que se prueban como primer trial (punto de partida).
    Devuelve (filas registradas en esta corrida, estudios, final de cada modelo), donde cada final es
    {"pipe", "params", "cfg", "estudio", "fila"} con el pipeline reentrenado en todo train."""
    desconocidos = set(modelos) - set(espacios)
    if desconocidos:
        raise ValueError(f"Modelos desconocidos: {sorted(desconocidos)}. Opciones: {list(espacios)}")

    df = cargar_datos(ruta_datos)  # sin test
    X_tr, y_tr = particion(df, "train")
    X_va, y_va = particion(df, "val")
    sha = huella(ruta_datos)[:12]
    corrida = datetime.now().isoformat(timespec="seconds")
    # test_size fijo: la ventana de entrenamiento más chica es la mitad de train
    cv = TimeSeriesSplit(n_splits=n_splits, test_size=len(X_tr) // (2 * n_splits))
    filas, estudios, finales = [], {}, {}

    def anotar(modelo, estudio, etapa, estado, params, cfg, seg, n_feat, **metricas):
        fila = {
            "corrida": corrida, "timestamp": datetime.now().isoformat(timespec="seconds"),
            "modelo": modelo, "estudio": estudio, "etapa": etapa, "estado": estado,
            "hiperparametros": json.dumps(params, sort_keys=True, default=str),
            "preprocesamiento": json.dumps(cfg, sort_keys=True, default=str) if cfg else None,
            "seg_entrenamiento": round(seg, 3), "n_features": n_feat,
            "n_train": len(X_tr), "n_val": len(X_va), "semilla": SEMILLA, "dataset_sha256": sha,
            # redondeo: con n_jobs=-1 el orden de suma cambia decimales lejanos (ruido de punto flotante)
            **{k: (round(v, 6) if isinstance(v, float) else v) for k, v in metricas.items()},
        }
        registrar_experimento(fila, ruta_logs)
        filas.append(fila)
        return fila

    # --- baselines: mismos folds del CV y mismo val que los modelos
    base = predicciones_baselines(df)
    for nombre in base:
        pred_tr = base.loc[X_tr.index, nombre]
        maes = [_metricas(y_tr.iloc[e], pred_tr.iloc[e])["mae"] for _, e in cv.split(X_tr)]
        m_val = _metricas(y_va, base.loc[X_va.index, nombre])
        anotar(nombre, None, "baseline", "completo", {}, None, 0.0, None,
               mae_cv=float(np.mean(maes)), std_cv=float(np.std(maes)), n_folds=len(maes),
               mae_val=m_val["mae"], rmse_val=m_val["rmse"], r2_val=m_val["r2"],
               mae_train=_metricas(y_tr, pred_tr)["mae"])
        if verbose:
            print(f"{nombre:26s} MAE cv = {np.mean(maes):8.1f}  MAE val = {m_val['mae']:8.1f}")

    # --- modelos
    for nombre in modelos:
        t_modelo = time.perf_counter()
        nombre_estudio = f"{nombre}_{etiqueta}_{corrida}"

        def anotar_cv(params, cfg, maes, seg, estado, n_feat, _n=nombre, _e=nombre_estudio):
            todo_nan = np.all(np.isnan(maes))
            anotar(_n, _e, "optuna_cv", estado, params, cfg, seg, n_feat,
                   mae_cv=np.nan if todo_nan else float(np.nanmean(maes)),
                   std_cv=np.nan if todo_nan else float(np.nanstd(maes)),
                   n_folds=int(np.sum(~np.isnan(maes))))

        estudio = optuna.create_study(
            direction="minimize",
            study_name=nombre_estudio,
            storage=f"sqlite:///{RUTA_OPTUNA_DB.as_posix()}",
            # multivariate + group: el espacio es condicional (cada selección tiene sus propios parámetros)
            sampler=optuna.samplers.TPESampler(seed=SEMILLA, multivariate=True, group=True),
            # no poda hasta tener n_startup trials completos ni antes del 2.º fold
            pruner=optuna.pruners.MedianPruner(n_startup_trials=n_startup, n_warmup_steps=1),
        )
        if encolar and encolar.get(nombre):
            estudio.enqueue_trial(encolar[nombre])
        estudio.optimize(_objetivo(nombre, X_tr, y_tr, cv, anotar_cv, espacios), n_trials=n_trials, timeout=timeout)
        estudios[nombre] = estudio

        # mejor config -> todo train -> validation
        params, cfg = sugerir(optuna.trial.FixedTrial(estudio.best_params), nombre, espacios)
        t0 = time.perf_counter()
        pipe = ajustar(nombre, params, cfg, X_tr, y_tr, espacios)
        seg = time.perf_counter() - t0
        m_val = _metricas(y_va, pipe.predict(X_va))
        fila = anotar(nombre, nombre_estudio, "optuna_val", "completo", params, cfg, seg, len(columnas_elegidas(pipe)),
                      mae_cv=estudio.best_value, std_cv=estudio.best_trial.user_attrs.get("std_cv", np.nan),
                      n_folds=n_splits, mae_val=m_val["mae"], rmse_val=m_val["rmse"], r2_val=m_val["r2"],
                      mae_train=_metricas(y_tr, pipe.predict(X_tr))["mae"])
        finales[nombre] = {"pipe": pipe, "params": params, "cfg": cfg, "estudio": nombre_estudio, "fila": fila}

        if verbose:
            estados = [t.state for t in estudio.trials]
            podados = sum(s == optuna.trial.TrialState.PRUNED for s in estados)
            print(f"{nombre:26s} MAE cv = {estudio.best_value:8.1f}  MAE val = {m_val['mae']:8.1f}  "
                  f"({len(estados)} trials, {podados} podados, {time.perf_counter() - t_modelo:.0f} s)")

    return pd.DataFrame(filas), estudios, finales

### Funciones de resumen y análisis
Se usan para el estudio general y para la búsqueda exhaustiva.

In [ ]:
def resumen(resultados: pd.DataFrame) -> pd.DataFrame:
    """Modelo final de cada estudio y cada baseline, ordenado por MAE de validación,
    con la mejora (%) respecto a la persistencia."""
    t = (resultados[resultados["etapa"].isin(["baseline", "optuna_val"])].set_index("modelo")
         [["mae_cv", "mae_val", "rmse_val", "r2_val", "mae_train", "n_features"]])
    ref = t.loc["baseline_persistencia"]
    t["mejora_cv_%"] = 100 * (1 - t["mae_cv"] / ref["mae_cv"])
    t["mejora_val_%"] = 100 * (1 - t["mae_val"] / ref["mae_val"])
    cols = ["mae_cv", "mae_val", "mejora_cv_%", "mejora_val_%", "rmse_val", "r2_val", "mae_train", "n_features"]
    return t[cols].sort_values("mae_val")


def configs_ganadoras(resultados: pd.DataFrame) -> pd.DataFrame:
    """Preprocesamiento elegido por cada estudio."""
    val = resultados[resultados["etapa"] == "optuna_val"].set_index("modelo")
    return pd.DataFrame({m: json.loads(val.loc[m, "preprocesamiento"]) for m in val.index}).T.fillna("-")


def graficar_estudios(estudios: dict[str, optuna.Study]) -> None:
    """Por estudio: historia, MAE acumulado por fold (los podados se cortan antes) e importancia de parámetros.
    MAE en escala log: algunas configuraciones extrapolan muy mal y sin log aplastan al resto."""
    for nombre, estudio in estudios.items():
        estados = pd.Series([t.state.name for t in estudio.trials]).value_counts().to_dict()
        print(f"=== {nombre}: {estados}")
        # cada función de optuna crea su propia figura y devuelve el Axes
        ax = optuna_plt.plot_optimization_history(estudio)
        ax.set_yscale("log")
        ax.set_title(f"{nombre}: historia (MAE cv)")
        ax = optuna_plt.plot_intermediate_values(estudio)
        ax.set_yscale("log")
        if ax.get_legend():
            ax.get_legend().remove()   # una entrada por trial: con muchos trials no se lee
        ax.set(title=f"{nombre}: MAE acumulado por fold (cortadas = podados)", xlabel="fold")
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", UserWarning)  # "pocos trials en algún régimen"
            try:
                ax = optuna_plt.plot_param_importances(estudio)
                ax.set_title(f"{nombre}: importancia de parámetros", loc="left")  # reemplaza el de optuna
            except (ValueError, RuntimeError) as e:  # p. ej. muy pocos trials completos
                print(f"  sin importancias: {e}")
        plt.show()


def mostrar_variables(finales: dict[str, dict]) -> None:
    for nombre, f in finales.items():
        cols = columnas_elegidas(f["pipe"])
        total = len(f["pipe"][0].feature_names_in_)
        externas = set(columnas_externas(cols))
        print(f"\n=== {nombre} ({len(cols)} de {total} variables: {len(externas)} externas fijas "
              f"+ {len(cols) - len(externas)} elegidas)")
        print("todas (sin selección)" if len(cols) == total else [c for c in cols if c not in externas])

### Correr el estudio general

In [ ]:
res, estudios, finales = comparar_modelos()
print(f"\nLog: {RUTA_LOGS}\nEstudios: {RUTA_OPTUNA_DB}")

## Comparación con los baselines

`mae_cv`: MAE medio en los folds de train (el que optimizó Optuna). `mae_val`: modelo final reentrenado en todo
train y medido en validation. `mejora_*`: cuánto baja el MAE respecto a la persistencia (positivo = le gana).

In [ ]:
tabla = resumen(res)
tabla.round(2)

### Mejor configuración de preprocesamiento por modelo

In [ ]:
configs_ganadoras(res)

In [ ]:
# MAE de validación por modelo; los baselines van como referencia vertical
TINTA, TINTA_2, MUDO, GRILLA, SERIE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#2a78d6"

modelos_val = tabla.drop(index=[i for i in tabla.index if i.startswith("baseline")]).sort_values("mae_val")
plt.style.use("default")   # por si ya corrieron los gráficos de optuna (dejan el estilo ggplot)
fig, ax = plt.subplots(figsize=(8, 0.6 * len(modelos_val) + 1.6))
ax.barh(modelos_val.index, modelos_val["mae_val"], height=0.5, color=SERIE, zorder=2)
for y_i, v in enumerate(modelos_val["mae_val"]):
    ax.text(v, y_i, f"{v:,.0f}  ", va="center", ha="right", color="#ffffff", fontsize=9, zorder=4)

# etiqueta de cada baseline a la derecha de su línea: una arriba y otra abajo para que no se pisen
n = len(modelos_val)
for (nombre, estilo), y_txt, va in [(("baseline_persistencia", "--"), -0.6, "bottom"),
                                    (("baseline_media_movil_7d", ":"), n - 0.45, "top")]:
    v = tabla.loc[nombre, "mae_val"]
    ax.axvline(v, color=TINTA_2, linestyle=estilo, linewidth=1.5, zorder=1)  # detrás de las barras
    ax.text(v, y_txt, f" {nombre.removeprefix('baseline_').replace('_', ' ')}: {v:,.0f}",
            ha="left", va=va, fontsize=8, color=TINTA_2)
ax.set_ylim(n + 0.1, -0.75)

ax.set_xlabel("MAE en validation (kW) - menor es mejor", color=TINTA_2)
ax.set_xlim(0, max(modelos_val["mae_val"].max(), tabla["mae_val"].max()) * 1.18)
ax.grid(axis="x", color=GRILLA, linewidth=0.8, zorder=0)
ax.tick_params(colors=MUDO, labelcolor=TINTA)
for lado in ("top", "right", "left"):
    ax.spines[lado].set_visible(False)
ax.spines["bottom"].set_color(MUDO)
ax.set_title("Modelos vs baselines", loc="left", color=TINTA, pad=12)
plt.tight_layout()
plt.show()

## Análisis del proceso

Por modelo: historia de la optimización, curvas por fold (las que terminan antes son los trials podados) e
importancia de cada parámetro (¿cuánto pesa el transformador, el scaler o la selección frente a los hiperparámetros?).
Los parámetros condicionales (`k`, `frac_rfe`, `alpha_sel`, ...) se estiman solo con los trials que los usaron: con pocos trials son ruidosos.

In [ ]:
graficar_estudios(estudios)

### Variables que usa cada modelo final

In [ ]:
mostrar_variables(finales)

## ¿TensorFlow para analizar el proceso? (evaluación, no implementado)

**Como modelo.** Con ~850 días de train el MLP de scikit-learn alcanza: una red en Keras no tiene más datos de
los que aprender. Sumaría una dependencia pesada (~500 MB) y en Windows nativo TensorFlow solo corre en CPU
(el soporte de GPU terminó en la 2.10). Lo que sí aportaría: pérdida MAE directa (el MLP de sklearn optimiza MSE) y
**pruning por época** con `optuna_integration.TFKerasPruningCallback`, más fino que el pruning por fold de acá.

**Para analizar el proceso.** TensorBoard (pestaña HParams, vía `optuna_integration.TensorBoardCallback`) mostraría
los trials y sus parámetros, pero es lo mismo que ya dan los gráficos de arriba y `optuna-dashboard`, que lee
directo `results/optuna.db` sin TensorFlow:

```
pip install optuna-dashboard
optuna-dashboard sqlite:///results/optuna.db
```

**Recomendación:** no agregar TensorFlow por ahora. Reconsiderarlo solo si se suma una red en Keras (por ejemplo una
LSTM sobre ventanas de días); en ese caso usar `TFKerasPruningCallback` y TensorBoard para las curvas de entrenamiento.

## Búsqueda exhaustiva: lasso y MLP

En el estudio general varios óptimos quedaron **en el borde** del espacio (lasso: `alpha` cerca del tope;
MLP: `k` y `n_rfe` en el mínimo, `alpha` cerca del tope, `learning_rate_init` cerca del mínimo), así que acá se
amplían los rangos, se usan más trials y el MLP prueba arquitecturas de hasta 4 capas ocultas:

| | Estudio general | Búsqueda exhaustiva |
|---|---|---|
| lasso `alpha` | 0.1 – 1e3 | 1 – 1e5 |
| lasso `k` del filtro | 10 – 150 | 10 – 300 |
| MLP capas | 32, 64, 64-32, 128-64 | 1–4 capas × {16, 32, 64, 128, 256} neuronas × forma `constante` / `embudo` (cada capa la mitad, mín. 8) |
| MLP `alpha` | 1e-4 – 10 | 1e-4 – 100 |
| MLP `learning_rate_init` | 1e-3 – 1e-2 | 1e-4 – 1e-2 |
| MLP `batch_size` | 32, 64, 128 | 16 – 256 |
| MLP `k` / `n_rfe` | 10–150 / mín. 5 | 5–100 / mín. 3 (`frac_rfe` hasta 1.0) |
| trials por modelo | 60 | 150 |

**Punto de partida:** el mejor trial del último estudio general de cada modelo (leído de `experiment_logs.csv`)
se encola como primer trial. Si ese estudio general es de la misma versión del pipeline, la búsqueda exhaustiva nunca
termina peor que la general en CV; si es de antes de agregar las variables externas fijas, es solo un punto de
partida razonable (la misma config, ahora con las externas).

In [ ]:
def capas_mlp(n_capas: int, neuronas: int, forma: str) -> tuple[int, ...]:
    """constante: 64-64-64 | embudo: cada capa la mitad de la anterior, mínimo 8 (128-64-32-16)."""
    return tuple(neuronas if forma == "constante" else max(8, neuronas // 2 ** i) for i in range(n_capas))


_NEURONAS = [16, 32, 64, 128, 256]
_BATCH = [16, 32, 64, 128, 256]


def _espacio_lasso_exh(t: optuna.Trial) -> dict:
    return {"alpha": t.suggest_float("alpha", 1.0, 1e5, log=True)}


def _espacio_mlp_exh(t: optuna.Trial) -> dict:
    capas = capas_mlp(t.suggest_int("n_capas", 1, 4), t.suggest_categorical("neuronas", _NEURONAS),
                      t.suggest_categorical("forma", ["constante", "embudo"]))
    return {
        "hidden_layer_sizes": capas,
        "activation": t.suggest_categorical("activation", ["relu", "tanh"]),
        "alpha": t.suggest_float("alpha", 1e-4, 100.0, log=True),
        "learning_rate_init": t.suggest_float("learning_rate_init", 1e-4, 1e-2, log=True),
        "batch_size": t.suggest_categorical("batch_size", _BATCH),
    }


ESPACIOS_EXH: dict[str, dict] = {
    "lasso": {**ESPACIOS["lasso"], "espacio": _espacio_lasso_exh,
              "prep": lambda t: espacio_prep(t, k=(10, 300, 10))},
    "mlp": {**ESPACIOS["mlp"], "espacio": _espacio_mlp_exh,
            "prep": lambda t: espacio_prep(t, k=(5, 100, 5), frac_rfe=(0.2, 1.0), n_rfe_min=3)},
}


def mejor_previo(modelo: str, ruta_logs: str | Path = RUTA_LOGS) -> dict | None:
    """Mejor config del último estudio general de `modelo`, traducida a los parámetros del espacio
    exhaustivo (para encolarla). None si no hay estudio previo o no se puede representar."""
    if not Path(ruta_logs).exists():
        return None
    log = pd.read_csv(ruta_logs)
    filas = log[(log["modelo"] == modelo) & (log["etapa"] == "optuna_val")]
    if "estudio" in log:  # las corridas viejas no tienen estudio: cuentan como generales
        est = filas["estudio"].fillna("").astype(str)
        filas = filas[(est == "") | est.str.contains("_general_")]
    if filas.empty:
        return None
    p, cfg = json.loads(filas.iloc[-1]["hiperparametros"]), json.loads(filas.iloc[-1]["preprocesamiento"])

    params = {k: v for k, v in cfg.items() if k != "n_rfe"}
    if "n_rfe" in cfg:
        params["frac_rfe"] = (cfg["n_rfe"] + 0.5) / cfg["k"]   # int(k * frac) vuelve a dar n_rfe
    params["alpha"] = p["alpha"]
    if modelo == "mlp":
        capas = tuple(p["hidden_layer_sizes"])
        forma = next((f for f in ("constante", "embudo") if capas_mlp(len(capas), capas[0], f) == capas), None)
        if forma is None or capas[0] not in _NEURONAS or p["batch_size"] not in _BATCH:
            print(f"{modelo}: la config previa {capas} no entra en el espacio exhaustivo, no se encola")
            return None
        params |= {"n_capas": len(capas), "neuronas": capas[0], "forma": forma,
                   **{k: p[k] for k in ("activation", "learning_rate_init", "batch_size")}}
    return params


encolar = {m: mejor_previo(m) for m in MODELOS_EXH}
encolar

In [ ]:
res_exh, estudios_exh, finales_exh = comparar_modelos(
    MODELOS_EXH, n_trials=N_TRIALS_EXH, timeout=TIMEOUT_EXH, espacios=ESPACIOS_EXH,
    etiqueta="exhaustivo", encolar=encolar, n_startup=10,
)

# control: con los mismos folds y la misma versión del pipeline, el trial encolado reproduce el MAE cv
# del mejor trial del estudio general
for m, est in estudios_exh.items():
    if encolar.get(m):
        t0 = est.trials[0]
        valor = f"{t0.value:.1f}" if t0.value is not None else "-"
        print(f"{m}: trial encolado (mejor del general) MAE cv = {valor} ({t0.state.name}); "
              f"mejor exhaustivo = {est.best_value:.1f}")

### Exhaustiva vs general vs baselines

In [ ]:
comp = resumen(res_exh).rename(index=lambda m: m if m.startswith("baseline") else f"{m} (exhaustivo)")
if "res" in globals():  # si en esta sesión también corrió el estudio general
    gen = resumen(res).loc[[m for m in MODELOS_EXH if m in set(res["modelo"])]]
    comp = pd.concat([comp, gen.rename(index=lambda m: f"{m} (general)")])
comp.sort_values("mae_val").round(2)

In [ ]:
configs_ganadoras(res_exh)

In [ ]:
for m, f in finales_exh.items():
    print(m, {k: v for k, v in f["params"].items() if k not in ESPACIOS_EXH[m]["fijos"]})
mostrar_variables(finales_exh)

In [ ]:
graficar_estudios(estudios_exh)

## Registro de modelos

Se guarda con joblib el pipeline final de cada estudio exhaustivo (mejor config por CV, reentrenado en todo train) y
se registra en `models/model_registry.json`. Se entrena **solo con train** para que val siga siendo una medición
limpia; el reentrenamiento con train + val queda para la evaluación final en test.

**Versionado semántico por modelo**, comparando contra la última versión registrada de ese modelo:

| Cambió | Versión |
|---|---|
| el dataset (sha256 de `dataset_final.csv`) | MAJOR → `v1.0.0` |
| el preprocesamiento / la selección o la arquitectura (capas, activación) | MINOR → `v0.1.0` |
| solo hiperparámetros | PATCH → `v0.0.1` |
| nada | no se registra |

La primera versión es `v0.0.0`. Para cargar un modelo desde cualquier script:
`sys.path.insert(0, "src"); joblib.load("models/<modelo>_v0.0.0.joblib")` (necesita `src/feature_selection.py`).

In [ ]:
_ARQUITECTURA = ("hidden_layer_sizes", "activation")   # cambiar esto sube MINOR


def _normalizar(x):
    """Mismo formato que queda en el JSON (tuplas -> listas), para comparar con lo registrado."""
    return json.loads(json.dumps(x, sort_keys=True, default=str))


def _relativa_root(ruta: Path) -> str:
    try:
        return ruta.resolve().relative_to(ROOT).as_posix()
    except ValueError:
        return ruta.resolve().as_posix()


def version_siguiente(previa: dict | None, nueva: dict) -> str | None:
    """vMAJOR.MINOR.PATCH según qué cambió respecto de `previa`; None si no cambió nada."""
    if previa is None:
        return "v0.0.0"
    mayor, menor, parche = map(int, previa["version"].removeprefix("v").split("."))
    if previa["dataset"]["sha256"] != nueva["dataset"]["sha256"]:
        return f"v{mayor + 1}.0.0"
    if previa["preprocesamiento"] != nueva["preprocesamiento"] or previa["arquitectura"] != nueva["arquitectura"]:
        return f"v{mayor}.{menor + 1}.0"
    if previa["hiperparametros"] != nueva["hiperparametros"]:
        return f"v{mayor}.{menor}.{parche + 1}"
    return None


def registrar_modelo(nombre: str, final: dict, ref_persistencia: float,
                     registro: Path = RUTA_REGISTRO_MODELOS, carpeta: Path = RUTA_MODELOS,
                     ruta_datos: Path = RUTA_FINAL) -> dict | None:
    """Guarda el pipeline con joblib y agrega una entrada al registro si algo cambió.
    `final` es una entrada de `finales_exh` (pipe, params, cfg, estudio, fila de validación)."""
    entradas = cargar_registro(registro)
    params, fila = _normalizar(final["params"]), final["fila"]
    nueva = {
        "modelo": nombre,
        "arquitectura": {k: params[k] for k in _ARQUITECTURA if k in params},
        "hiperparametros": {k: v for k, v in params.items() if k not in _ARQUITECTURA},
        # las externas fijas son parte del preprocesamiento: si cambia la lista, sube MINOR
        "preprocesamiento": {**_normalizar(final["cfg"]),
                             "externas_siempre": columnas_externas(final["pipe"][0].feature_names_in_)},
        "dataset": {"archivo": _relativa_root(Path(ruta_datos)), "sha256": huella(ruta_datos)},
    }
    previa = next((e for e in reversed(entradas) if e["modelo"] == nombre), None)
    version = version_siguiente(previa, nueva)
    if version is None:
        print(f"{nombre}: sin cambios respecto de {previa['version']}, no se registra.")
        return None

    carpeta.mkdir(parents=True, exist_ok=True)
    archivo = carpeta / f"{nombre}_{version}.joblib"
    pipe = final["pipe"].set_params(memory=None)   # no guardar la ruta de la caché
    joblib.dump(pipe, archivo, compress=3)

    cols = columnas_elegidas(pipe)
    entrada = {
        "modelo": nombre, "version": version,
        "archivo": _relativa_root(archivo), "sha256": huella(archivo),
        "fecha": datetime.now().isoformat(timespec="seconds"),
        "version_previa": previa["version"] if previa else None,
        "estudio_optuna": final["estudio"], "corrida": fila["corrida"],
        **{k: nueva[k] for k in ("dataset", "arquitectura", "hiperparametros", "preprocesamiento")},
        "metricas": {
            **{k: fila.get(k) for k in ("mae_cv", "std_cv", "mae_val", "rmse_val", "r2_val", "mae_train")},
            "mejora_val_vs_persistencia_%": round(100 * (1 - fila["mae_val"] / ref_persistencia), 3),
        },
        "entrenado_con": {"split": "train", "n_filas": fila["n_train"]},
        "n_features": len(cols), "columnas": cols, "semilla": SEMILLA,
        "entorno": {"python": platform.python_version(), "scikit-learn": sklearn.__version__,
                    "numpy": np.__version__, "pandas": pd.__version__, "joblib": joblib.__version__,
                    "optuna": optuna.__version__},
        "cargar_con": f"sys.path.insert(0, 'src'); joblib.load('{_relativa_root(archivo)}')",
    }
    guardar_registro(entradas + [entrada], registro)
    print(f"{nombre}: registrado {version} -> {entrada['archivo']} (MAE val {fila['mae_val']:.1f})")
    return entrada

In [ ]:
ref_persistencia = res_exh.set_index("modelo").loc["baseline_persistencia", "mae_val"]
registrados = {m: registrar_modelo(m, finales_exh[m], ref_persistencia) for m in MODELOS_EXH}
pd.DataFrame([{"modelo": e["modelo"], "version": e["version"], "archivo": e["archivo"], **e["metricas"]}
              for e in cargar_registro(RUTA_REGISTRO_MODELOS)])

## Comparación de residuos

Residuo = real − predicho, en validation, para la última versión registrada de lasso y MLP (cargadas **desde el
registro**, así se comprueba que los `.joblib` se abren y coinciden con su sha256) y la persistencia como referencia.
Residuo positivo = el modelo se quedó corto (predijo menos consumo del que hubo).

In [ ]:
registro_modelos = cargar_registro(RUTA_REGISTRO_MODELOS)
ultimas = {e["modelo"]: e for e in registro_modelos}   # la última versión de cada modelo

df_val = cargar_datos()
X_va, y_va = particion(df_val, "val")
predicciones = {}
for m in MODELOS_EXH:
    e = ultimas[m]
    archivo = Path(e["archivo"]) if Path(e["archivo"]).is_absolute() else ROOT / e["archivo"]
    if huella(archivo) != e["sha256"]:
        raise ValueError(f"{archivo} no coincide con el sha256 registrado")
    predicciones[f"{m} {e['version']}"] = joblib.load(archivo).predict(X_va)
predicciones["persistencia"] = predicciones_baselines(df_val).loc[X_va.index, "baseline_persistencia"].to_numpy()

residuos = pd.DataFrame({k: y_va.to_numpy() - v for k, v in predicciones.items()}, index=X_va.index)
predichos = pd.DataFrame(predicciones, index=X_va.index)

UMBRAL = 3000  # kW
pd.DataFrame({
    "sesgo (media)": residuos.mean(),
    "desvío": residuos.std(),
    "MAE": residuos.abs().mean(),
    "p95 |residuo|": residuos.abs().quantile(0.95),
    f"% días |residuo| > {UMBRAL}": 100 * (residuos.abs() > UMBRAL).mean(),
}).round(1)

In [ ]:
# colores: slots categóricos 1-3 de la paleta de referencia (validados con validate_palette.js)
COLORES = dict(zip(residuos.columns, ["#2a78d6", "#eb6834", "#1baf7a"]))
TINTA, TINTA_2, MUDO, GRILLA = "#0b0b0b", "#52514e", "#898781", "#e1e0d9"


def _estilo(ax):
    ax.grid(color=GRILLA, linewidth=0.8, zorder=0)
    ax.tick_params(colors=MUDO, labelcolor=TINTA_2, labelsize=8)
    for lado in ("top", "right"):
        ax.spines[lado].set_visible(False)
    for lado in ("left", "bottom"):
        ax.spines[lado].set_color(MUDO)


n = len(residuos.columns)
plt.style.use("default")   # los gráficos de optuna dejan puesto el estilo ggplot (fondo gris)
fig = plt.figure(figsize=(13, 1.9 * n + 4.6))
grid = fig.add_gridspec(2, 1, height_ratios=[n, 2.2], hspace=0.22)
arriba = grid[0].subgridspec(n, 1, hspace=0.12)
abajo = grid[1].subgridspec(1, 2, wspace=0.18)
lim = residuos.abs().quantile(0.995).max() * 1.1

# arriba: residuo en el tiempo, un panel por modelo (mismo eje y)
ax0 = None
for i, col in enumerate(residuos.columns):
    ax = fig.add_subplot(arriba[i], sharex=ax0, sharey=ax0)
    ax0 = ax0 or ax
    ax.axhline(0, color=TINTA_2, linewidth=1, zorder=1)
    ax.plot(residuos.index, residuos[col], color=COLORES[col], linewidth=1.2, zorder=2)
    ax.set_ylim(-lim, lim)
    _estilo(ax)
    ax.text(0.005, 0.95, f"{col}  ·  MAE {residuos[col].abs().mean():,.0f}  ·  sesgo {residuos[col].mean():+,.0f} kW",
            transform=ax.transAxes, va="top", fontsize=9, color=TINTA, zorder=3,
            bbox={"facecolor": "white", "edgecolor": "none", "alpha": 0.85, "pad": 2})
    if i < n - 1:
        ax.tick_params(labelbottom=False)
ax0.set_title("Residuos en validation (real − predicho, kW)", loc="left", color=TINTA)

# abajo izquierda: distribución
ax = fig.add_subplot(abajo[0])
cajas = ax.boxplot([residuos[c] for c in residuos.columns], orientation="horizontal", widths=0.5,
                   patch_artist=True, showfliers=True, medianprops={"color": TINTA, "linewidth": 1.5},
                   flierprops={"marker": "o", "markersize": 3, "alpha": 0.5})
for caja, col in zip(cajas["boxes"], residuos.columns):
    caja.set(facecolor=COLORES[col], alpha=0.85, edgecolor=COLORES[col])
ax.set_yticks(range(1, n + 1), residuos.columns)
ax.axvline(0, color=TINTA_2, linewidth=1)
ax.invert_yaxis()
_estilo(ax)
ax.set_title("Distribución", loc="left", color=TINTA, fontsize=10)
ax.set_xlabel("residuo (kW)", color=TINTA_2, fontsize=9)

# abajo derecha: residuo vs predicho (¿el error depende del nivel de consumo?)
ax = fig.add_subplot(abajo[1])
for col in residuos.columns:
    ax.scatter(predichos[col], residuos[col], s=12, alpha=0.55, color=COLORES[col], label=col,
               edgecolors="white", linewidths=0.4, zorder=2)
ax.axhline(0, color=TINTA_2, linewidth=1, zorder=1)
_estilo(ax)
ax.set_title("Residuo vs predicho", loc="left", color=TINTA, fontsize=10)
ax.set_xlabel("predicho (kW)", color=TINTA_2, fontsize=9)
ax.legend(frameon=False, fontsize=8, loc="upper right")
plt.show()